# Modelagem de Machine Learning

Treinamento e avaliação de modelos para predição de churn.

Este notebook contém o treinamento dos modelos de ML.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')

from src.data.load_data import generate_synthetic_data
from src.features.build_features import create_features
from src.models.train_model import ModelTrainer
import yaml

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

df = generate_synthetic_data(n_samples=5000)
df_processed = create_features(df, config, fit=True)

print(f"Dados carregados: {df_processed.shape}")

In [ ]:
target_col = config['model']['target_column']
exclude_cols = [config['features']['id_column'], target_col, 'tenure_group']
feature_cols = [c for c in df_processed.columns if c not in exclude_cols]

X = df_processed[feature_cols]
y = df_processed[target_col]

print(f"Features: {len(feature_cols)}")
print(f"\nDistribuição do target:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean():.2%}")

In [ ]:
trainer = ModelTrainer(config)
X_train, X_test, y_train, y_test = trainer.split_data(X, y)
results = trainer.train_all(X_train, y_train, X_test, y_test)

In [ ]:
importance = trainer.get_feature_importance()
print("Top 10 Features Mais Importantes:")
print(importance.head(10).to_string(index=False))

In [ ]:
top15 = importance.head(15).sort_values('importance', ascending=True)

fig = px.bar(
    top15, y='feature', x='importance', orientation='h',
    text_auto='.4f', color='importance', color_continuous_scale='Viridis'
)
fig.update_layout(
    title='Top 15 Features Mais Importantes',
    xaxis_title='Importância',
    yaxis_title='Feature',
    showlegend=False
)
fig.show()

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_result = results[best_model_name]

print(f"\nMELHOR MODELO: {best_model_name.upper()}")
print(f"   ROC AUC: {best_result['roc_auc']:.4f}")
print(f"   F1 Score: {best_result['f1']:.4f}")
print(f"   Precision: {best_result['precision']:.4f}")
print(f"   Recall: {best_result['recall']:.4f}")

In [ ]:
models = list(results.keys())

fig = make_subplots(rows=1, cols=2, subplot_titles=('Métricas', 'ROC AUC'))

for metric in ['accuracy', 'precision', 'recall', 'f1']:
    values = [results[m].get(metric, 0) for m in models]
    fig.add_trace(
        go.Bar(name=metric, x=models, y=values, texttemplate='%{y:.3f}'),
        row=1, col=1
    )

auc_values = [results[m].get('roc_auc', 0) for m in models]
fig.add_trace(
    go.Bar(name='ROC AUC', x=models, y=auc_values,
           texttemplate='%{y:.3f}', marker_color='#3498db'),
    row=1, col=2
)

fig.update_layout(title='Comparação de Modelos', barmode='group', height=500)
fig.show()

In [ ]:
cm = np.array(best_result['confusion_matrix'])

fig = px.imshow(
    cm, text_auto=True, color_continuous_scale='Blues',
    labels=dict(x='Predito', y='Real'),
    x=['Não Churn', 'Churn'],
    y=['Não Churn', 'Churn']
)
fig.update_layout(title=f"Matriz de Confusão - {best_model_name.upper()}")
fig.show()

print("\nClassification Report:")
print(best_result['classification_report'])

In [ ]:
model_path = trainer.save_model('../models/')
trainer.save_results('../reports/')

print(f"\nModelo salvo em: {model_path}")